<a href="https://colab.research.google.com/github/rayana11-blip/Technology_Usage_Analytics/blob/main/Github2_dash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import os

p = 'Technology_Usage_Survey_Responses_English.csv'

df = pd.read_csv(p)

print("Dataset successfully loaded!")
print("Dataset shape:", df.shape)
print("Column names:", df.columns.tolist())

display(df.head())

Dataset successfully loaded!
Dataset shape: (55, 6)
Column names: ['ID', 'Frequently Used Devices', 'Purpose of Use', 'User Age', 'Time Spent per Day (hours)', 'Monthly Data Usage (GB)']


,ID,Frequently Used Devices,Purpose of Use,User Age,Time Spent per Day (hours),Monthly Data Usage (GB)
0,ID001,Smartphone,Entertainment,4 years,8 hours,5 GB
1,ID002,Smartphone,Communication tool & learning media,18,7,50 GB
2,ID003,"Smartphone, Laptop, Tablet",Entertainment,18 years,6 hours,10 GB
3,ID004,Smartphone,Communication,18,9 hours,20 GB
4,ID005,Smartphone,Communication,18,6,Uses Wi-Fi


In [13]:

# ================================================================
# TECHNOLOGY USAGE INTELLIGENCE DASHBOARD
# CYBER NEON EDITION
# GOOGLE COLAB - COMPLETE VERSION
# ================================================================

# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import re
import io

from IPython.display import display, HTML, clear_output
from google.colab import files

# ================================================================
# 2. UPLOAD DATASET
# ================================================================

uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload your survey CSV file.")

file_name = next(iter(uploaded))
df_raw = pd.read_csv(file_name)

required_columns = [
    "ID",
    "Frequently Used Devices",
    "Purpose of Use",
    "User Age",
    "Time Spent per Day (hours)",
    "Monthly Data Usage (GB)"
]

missing_columns = [
    c for c in required_columns
    if c not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {df_raw.columns.tolist()}"
    )

print("Dataset successfully loaded!")
print("Dataset shape:", df_raw.shape)

# ================================================================
# 3. DATA CLEANING
# ================================================================

def convert_numeric(value):

    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if text == "":
        return np.nan

    if "wifi" in text or "wi-fi" in text:
        return np.nan

    if text in ["≤10", "<=10", "6-apr"]:
        return np.nan

    if text.startswith("-"):
        return np.nan

    numbers = re.findall(r"\d+(?:\.\d+)?", text)

    if not numbers:
        return np.nan

    # Convert ranges to midpoint
    if re.search(r"\d\s*-\s*\d", text) and len(numbers) >= 2:
        return (float(numbers[0]) + float(numbers[1])) / 2

    return float(numbers[0])


data = pd.DataFrame({

    "ID": df_raw["ID"].astype(str),

    "Device": (
        df_raw["Frequently Used Devices"]
        .fillna("Not specified")
        .astype(str)
        .str.strip()
    ),

    "Purpose": (
        df_raw["Purpose of Use"]
        .fillna("Not specified")
        .astype(str)
        .str.strip()
    ),

    "Age": df_raw["User Age"].apply(convert_numeric),

    "Daily Usage (Hours)": (
        df_raw["Time Spent per Day (hours)"]
        .apply(convert_numeric)
    ),

    "Monthly Data (GB)": (
        df_raw["Monthly Data Usage (GB)"]
        .apply(convert_numeric)
    )

})

# Previously agreed cleaning decision:
# Age response "4 years" is treated as missing.

age_text = (
    df_raw["User Age"]
    .astype(str)
    .str.strip()
    .str.lower()
)

data.loc[
    age_text.isin(["4", "4 years", "4 year"]),
    "Age"
] = np.nan

numeric_variables = [
    "Age",
    "Daily Usage (Hours)",
    "Monthly Data (GB)"
]

print("Data cleaning completed.")
print("Total respondents:", len(data))
display(data.head())

# ================================================================
# 4. CYBER NEON COLOR PALETTE
# ================================================================

NEON_COLORS = [
    "#00E5FF",
    "#B65CFF",
    "#39F5A5",
    "#FF4FA3",
    "#FFD166"
]

BG_COLOR = "#080D18"
PANEL_COLOR = "#111B2D"
CARD_COLOR = "#142238"
CYAN = "#00E5FF"
PURPLE = "#B65CFF"
GREEN = "#39F5A5"
PINK = "#FF4FA3"
WHITE = "#F2F7FF"
MUTED = "#91A4C2"
GRID = "#263B55"

# ================================================================
# 5. DASHBOARD CSS DESIGN
# ================================================================

display(HTML("""
<style>

body {
    background: #080D18 !important;
}

.jp-OutputArea,
.output,
.output_wrapper {
    background: #080D18 !important;
}

.dashboard-header {
    background:
        radial-gradient(
            circle at 85% 20%,
            rgba(0,229,255,0.22),
            transparent 30%
        ),
        radial-gradient(
            circle at 15% 100%,
            rgba(182,92,255,0.18),
            transparent 30%
        ),
        linear-gradient(
            120deg,
            #080D18,
            #102039,
            #10172A
        );

    border: 1px solid #00E5FF;
    border-radius: 17px;
    padding: 27px;
    margin: 10px 0 20px 0;

    box-shadow:
        0 0 18px rgba(0,229,255,0.13),
        inset 0 0 20px rgba(0,229,255,0.04);
}

.dashboard-header h1 {
    color: #FFFFFF;
    font-size: 27px;
    letter-spacing: 2px;
    font-weight: bold;
    margin-bottom: 8px;
}

.dashboard-header p {
    color: #00E5FF;
    font-size: 11px;
    letter-spacing: 2px;
}

.dashboard-header .subtitle {
    color: #A8B9D5;
    font-size: 12px;
}

.kpi-grid {
    display: grid;
    grid-template-columns: repeat(4,minmax(0,1fr));
    gap: 13px;
    margin: 17px 0;
}

.kpi-card {
    background: linear-gradient(
        145deg,
        #182942,
        #0D1729
    );

    border: 1px solid #29415F;
    border-radius: 13px;
    padding: 17px;

    box-shadow: 0 0 13px rgba(0,229,255,0.06);
}

.kpi-card:nth-child(1) {
    border-top: 3px solid #00E5FF;
}

.kpi-card:nth-child(2) {
    border-top: 3px solid #B65CFF;
}

.kpi-card:nth-child(3) {
    border-top: 3px solid #39F5A5;
}

.kpi-card:nth-child(4) {
    border-top: 3px solid #FF4FA3;
}

.kpi-label {
    color: #A7B7CF;
    font-size: 10px;
    letter-spacing: 1px;
    font-weight: bold;
}

.kpi-value {
    color: #FFFFFF;
    font-size: 25px;
    font-weight: bold;
    margin-top: 8px;
}

.kpi-note {
    color: #8194B1;
    font-size: 10px;
    margin-top: 5px;
}

.summary-panel {
    background: linear-gradient(
        110deg,
        #10243D,
        #192D49
    );

    border: 1px solid #00E5FF;
    border-radius: 12px;
    color: #FFFFFF;
    padding: 17px;
    margin: 15px 0;

    box-shadow: 0 0 12px rgba(0,229,255,0.08);
}

.section-title {
    color: #00E5FF;
    font-size: 15px;
    font-weight: bold;
    letter-spacing: 1px;
    margin: 20px 0 12px;
}

.dashboard-footer {
    background: #101B2D;
    color: #91A4C2;
    border: 1px solid #263B55;
    padding: 13px;
    border-radius: 10px;
    margin-top: 18px;
    font-size: 11px;
}

.widget-label {
    color: #00E5FF !important;
}

</style>
"""))

# ================================================================
# 6. DASHBOARD HEADER
# ================================================================

display(HTML("""
<div class="dashboard-header">

    <h1>TECHNOLOGY USAGE INTELLIGENCE</h1>

    <p>
        DIGITAL BEHAVIOR MONITORING SYSTEM
    </p>

    <div class="subtitle">
        Statistical Analysis
        &nbsp; | &nbsp;
        Data Visualization
        &nbsp; | &nbsp;
        Technology Behavior
    </div>

    <div style="
        color:#00E5FF;
        font-size:11px;
        margin-top:18px;
        letter-spacing:2px;
    ">
        ✦ DIGITAL DATA INSIGHT ✦ ANALYTICS SYSTEM ✦
    </div>

</div>
"""))

# ================================================================
# 7. INTERACTIVE DASHBOARD CONTROLS
# ================================================================

variable_widget = widgets.Dropdown(

    options=[
        ("Age (Years)", "Age"),
        ("Daily Usage (Hours)", "Daily Usage (Hours)"),
        ("Monthly Data Usage (GB)", "Monthly Data (GB)"),
        ("Frequently Used Devices", "Device"),
        ("Purpose of Use", "Purpose")
    ],

    value="Age",
    description="Variable:",
    layout=widgets.Layout(width="280px")
)


chart_widget = widgets.Dropdown(

    options=[
        ("Ranking", "ranking"),
        ("Histogram", "histogram"),
        ("Boxplot", "boxplot"),
        ("Scatter Plot", "scatter"),
        ("Correlation Matrix", "correlation")
    ],

    value="ranking",
    description="Chart:",
    layout=widgets.Layout(width="250px")
)


limit_widget = widgets.Dropdown(

    options=[
        ("All Data", "all"),
        ("Top 10", 10),
        ("Top 5", 5)
    ],

    value="all",
    description="Display:",
    layout=widgets.Layout(width="200px")
)


device_filter = widgets.Dropdown(

    options=[
        "All Devices"
    ] + sorted(data["Device"].dropna().unique().tolist()),

    value="All Devices",
    description="Device:",
    layout=widgets.Layout(width="280px")
)


purpose_filter = widgets.Dropdown(

    options=[
        "All Purposes"
    ] + sorted(data["Purpose"].dropna().unique().tolist()),

    value="All Purposes",
    description="Purpose:",
    layout=widgets.Layout(width="280px")
)


controls = widgets.VBox([

    widgets.HTML("""
        <div style="
            color:#00E5FF;
            font-weight:bold;
            font-size:14px;
            letter-spacing:1px;
            padding:8px 0;
        ">
        ⚡ DASHBOARD CONTROL PANEL
        </div>
    """),

    widgets.HBox([
        variable_widget,
        chart_widget,
        limit_widget
    ]),

    widgets.HBox([
        device_filter,
        purpose_filter
    ])

])

display(controls)

# ================================================================
# 8. FILTER DATA
# ================================================================

def get_filtered_data():

    filtered = data.copy()

    if device_filter.value != "All Devices":

        filtered = filtered[
            filtered["Device"] == device_filter.value
        ]

    if purpose_filter.value != "All Purposes":

        filtered = filtered[
            filtered["Purpose"] == purpose_filter.value
        ]

    return filtered


# ================================================================
# 9. KPI STATISTICS
# ================================================================

def render_kpis(filtered, selected_variable):

    total = len(filtered)

    if selected_variable in numeric_variables:

        values = filtered[selected_variable].dropna()

        count = len(values)

        mean_value = (
            values.mean()
            if count > 0
            else np.nan
        )

        median_value = (
            values.median()
            if count > 0
            else np.nan
        )

        std_value = (
            values.std(ddof=1)
            if count > 1
            else np.nan
        )

        cards = [

            (
                "TOTAL RESPONDENTS",
                f"{total:,}",
                "Filtered respondents"
            ),

            (
                "VALID OBSERVATIONS",
                f"{count:,}",
                "Non-missing values"
            ),

            (
                "MEAN",
                f"{mean_value:.2f}" if pd.notna(mean_value) else "—",
                selected_variable
            ),

            (
                "STANDARD DEVIATION",
                f"{std_value:.2f}" if pd.notna(std_value) else "—",
                "Sample standard deviation"
            )

        ]

    else:

        values = filtered[selected_variable].dropna()

        counts = values.value_counts()

        mode = (
            counts.index[0]
            if len(counts) > 0
            else "—"
        )

        cards = [

            (
                "TOTAL RESPONDENTS",
                f"{total:,}",
                "Filtered respondents"
            ),

            (
                "VALID RESPONSES",
                f"{len(values):,}",
                "Available responses"
            ),

            (
                "MOST FREQUENT",
                str(mode)[:18],
                "Mode category"
            ),

            (
                "UNIQUE CATEGORIES",
                f"{values.nunique():,}",
                "Distinct categories"
            )

        ]

    html = "<div class='kpi-grid'>"

    for label, value, note in cards:

        html += f"""
        <div class="kpi-card">

            <div class="kpi-label">
                {label}
            </div>

            <div class="kpi-value">
                {value}
            </div>

            <div class="kpi-note">
                {note}
            </div>

        </div>
        """

    html += "</div>"

    display(HTML(html))


# ================================================================
# 10. SUMMARY INFORMATION
# ================================================================

def render_summary(filtered, selected_variable):

    if selected_variable in numeric_variables:

        values = filtered[
            ["ID", selected_variable]
        ].dropna()

        if len(values) > 0:

            highest = values.loc[
                values[selected_variable].idxmax()
            ]

            lowest = values.loc[
                values[selected_variable].idxmin()
            ]

            content = (

                f"<b style='color:#39F5A5;'>HIGHEST:</b> "
                f"{highest['ID']} = "
                f"{highest[selected_variable]:.2f}"

                f"&nbsp;&nbsp; | &nbsp;&nbsp;"

                f"<b style='color:#FF4FA3;'>LOWEST:</b> "
                f"{lowest['ID']} = "
                f"{lowest[selected_variable]:.2f}"

            )

        else:

            content = "No valid numeric observations."

    else:

        counts = filtered[selected_variable].value_counts()

        if len(counts) > 0:

            content = (

                f"<b style='color:#39F5A5;'>MOST FREQUENT:</b> "
                f"{counts.index[0]} ({counts.iloc[0]} responses)"

                f"&nbsp;&nbsp; | &nbsp;&nbsp;"

                f"<b style='color:#FF4FA3;'>LEAST FREQUENT:</b> "
                f"{counts.index[-1]} ({counts.iloc[-1]} responses)"

            )

        else:

            content = "No category data available."

    display(
        HTML(
            f"<div class='summary-panel'>{content}</div>"
        )
    )


# ================================================================
# 11. VISUALIZATION SETTINGS
# ================================================================

def style_axis(ax):

    ax.set_facecolor("#111B2D")

    ax.figure.set_facecolor("#080D18")

    ax.tick_params(
        colors="#DDE8FA",
        labelsize=9
    )

    ax.xaxis.label.set_color("#B8C9E4")
    ax.yaxis.label.set_color("#B8C9E4")
    ax.title.set_color("#FFFFFF")

    for spine in ax.spines.values():

        spine.set_color("#314763")

    ax.grid(
        color="#263B55",
        alpha=0.65,
        linestyle="--"
    )


# ================================================================
# 12. DATA VISUALIZATION
# ================================================================

def render_chart(
    filtered,
    selected_variable,
    chart_type,
    limit
):

    plt.close("all")

    fig, ax = plt.subplots(figsize=(11, 5))

    fig.patch.set_facecolor("#080D18")

    ax.set_facecolor("#111B2D")

    # ------------------------------------------------
    # CATEGORICAL VARIABLES
    # ------------------------------------------------

    if selected_variable in ["Device", "Purpose"]:

        counts = (
            filtered[selected_variable]
            .fillna("Not specified")
            .value_counts()
        )

        if limit != "all":

            counts = counts.head(int(limit))

        if chart_type in ["ranking", "histogram"]:

            bars = ax.barh(

                counts.index.astype(str)[::-1],

                counts.values[::-1],

                color=NEON_COLORS[
                    :len(counts)
                ][::-1],

                edgecolor=CYAN,

                linewidth=1.2

            )

            ax.set_xlabel("Number of Respondents")

            ax.set_title(
                f"{selected_variable} Distribution",
                loc="left",
                fontweight="bold",
                pad=15
            )

            for bar in bars:

                ax.text(

                    bar.get_width(),

                    bar.get_y() + bar.get_height()/2,

                    f" {int(bar.get_width())}",

                    va="center",

                    color=WHITE,

                    fontsize=9

                )

        elif chart_type == "boxplot":

            ax.text(
                0.5,
                0.5,
                "Boxplot requires a numeric variable.",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color=WHITE
            )

            ax.set_axis_off()

        else:

            ax.text(
                0.5,
                0.5,
                "Select a numeric variable for this visualization.",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color=WHITE
            )

            ax.set_axis_off()

    # ------------------------------------------------
    # NUMERIC VARIABLES
    # ------------------------------------------------

    else:

        values = filtered[
            selected_variable
        ].dropna()

        if chart_type == "ranking":

            sorted_values = (
                values.sort_values(ascending=False)
                .reset_index(drop=True)
            )

            if limit != "all":

                sorted_values = sorted_values.head(
                    int(limit)
                )

            ax.bar(

                np.arange(1, len(sorted_values)+1),

                sorted_values,

                color=CYAN,

                edgecolor="#FFFFFF",

                linewidth=0.7

            )

            ax.set_xlabel("Observation Rank")

            ax.set_ylabel(selected_variable)

            ax.set_title(
                f"{selected_variable} Ranking",
                loc="left",
                fontweight="bold",
                pad=15
            )

        elif chart_type == "histogram":

            ax.hist(

                values,

                bins="auto",

                color=PURPLE,

                edgecolor=CYAN,

                linewidth=1.2,

                alpha=0.9

            )

            if len(values) > 0:

                ax.axvline(

                    values.mean(),

                    color=GREEN,

                    linestyle="--",

                    linewidth=2,

                    label=f"Mean: {values.mean():.2f}"

                )

                ax.axvline(

                    values.median(),

                    color=PINK,

                    linestyle=":",

                    linewidth=2,

                    label=f"Median: {values.median():.2f}"

                )

                legend = ax.legend(
                    facecolor="#18263C",
                    edgecolor="#314763"
                )

                for text in legend.get_texts():

                    text.set_color(WHITE)

            ax.set_xlabel(selected_variable)

            ax.set_ylabel("Frequency")

            ax.set_title(
                f"{selected_variable} Distribution",
                loc="left",
                fontweight="bold",
                pad=15
            )

        elif chart_type == "boxplot":

            if len(values) > 0:

                ax.boxplot(

                    values,

                    vert=False,

                    patch_artist=True,

                    boxprops=dict(
                        facecolor="#233E5C",
                        color=CYAN
                    ),

                    medianprops=dict(
                        color=PINK,
                        linewidth=2.5
                    ),

                    whiskerprops=dict(
                        color=CYAN
                    ),

                    capprops=dict(
                        color=GREEN
                    ),

                    flierprops=dict(
                        marker="o",
                        markerfacecolor=PURPLE,
                        markeredgecolor=WHITE
                    )

                )

            ax.set_xlabel(selected_variable)

            ax.set_title(
                f"{selected_variable} Boxplot",
                loc="left",
                fontweight="bold",
                pad=15
            )

        elif chart_type == "scatter":

            valid = filtered[
                [
                    "Daily Usage (Hours)",
                    "Monthly Data (GB)"
                ]
            ].dropna()

            ax.scatter(

                valid["Daily Usage (Hours)"],

                valid["Monthly Data (GB)"],

                s=65,

                color=CYAN,

                edgecolor=WHITE,

                alpha=0.85

            )

            ax.set_xlabel("Daily Usage (Hours)")

            ax.set_ylabel("Monthly Data (GB)")

            ax.set_title(
                "Daily Usage vs Monthly Data",
                loc="left",
                fontweight="bold",
                pad=15
            )

        elif chart_type == "correlation":

            corr_data = filtered[
                numeric_variables
            ].corr(method="pearson")

            image = ax.imshow(

                corr_data,

                cmap="cool",

                vmin=-1,

                vmax=1

            )

            ax.set_xticks(
                range(len(corr_data.columns))
            )

            ax.set_yticks(
                range(len(corr_data.index))
            )

            ax.set_xticklabels(
                corr_data.columns,
                rotation=25,
                ha="right"
            )

            ax.set_yticklabels(
                corr_data.index
            )

            for i in range(len(corr_data.index)):

                for j in range(len(corr_data.columns)):

                    value = corr_data.iloc[i, j]

                    label = (
                        f"{value:.2f}"
                        if pd.notna(value)
                        else "—"
                    )

                    ax.text(
                        j,
                        i,
                        label,
                        ha="center",
                        va="center",
                        color="#FFFFFF",
                        fontweight="bold"
                    )

            fig.colorbar(
                image,
                ax=ax
            )

            ax.set_title(
                "Pearson Correlation Matrix",
                loc="left",
                fontweight="bold",
                pad=15
            )

    style_axis(ax)

    plt.tight_layout()

    display(fig)

    plt.close(fig)


# ================================================================
# 13. DATA SUMMARY TABLE
# ================================================================

def render_table(filtered, selected_variable):

    display(
        HTML("""
        <div class="section-title">
            RESPONDENT DATA EXPLORER
        </div>
        """)
    )

    if selected_variable in numeric_variables:

        table = filtered[
            [
                "ID",
                "Device",
                "Purpose",
                selected_variable
            ]
        ].copy()

        table = table.sort_values(
            selected_variable,
            ascending=False,
            na_position="last"
        )

    else:

        table = filtered[
            [
                "ID",
                "Device",
                "Purpose",
                "Age",
                "Daily Usage (Hours)",
                "Monthly Data (GB)"
            ]
        ].copy()

    display(
        table.reset_index(drop=True).head(100).style
        .format(na_rep="—")
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#142238"),
                    ("color", "#00E5FF"),
                    ("border", "1px solid #263B55")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("background-color", "#101B2D"),
                    ("color", "#F2F7FF"),
                    ("border", "1px solid #263B55")
                ]
            }
        ])
    )


# ================================================================
# 14. EXPORT FILTERED DATA
# ================================================================

def export_filtered_data():

    filtered = get_filtered_data()

    output = io.BytesIO()

    filtered.to_csv(
        output,
        index=False
    )

    files.download(
        "Technology_Usage_Filtered.csv"
    ) if False else None

    return filtered.to_csv(index=False).encode("utf-8")


# ================================================================
# 15. DASHBOARD UPDATE
# ================================================================

output_area = widgets.Output()

def update_dashboard(change=None):

    with output_area:

        clear_output(wait=True)

        filtered = get_filtered_data()

        selected_variable = variable_widget.value

        selected_chart = chart_widget.value

        selected_limit = limit_widget.value

        render_kpis(
            filtered,
            selected_variable
        )

        render_summary(
            filtered,
            selected_variable
        )

        display(
            HTML("""
            <div class="section-title">
                DATA VISUALIZATION
            </div>
            """)
        )

        render_chart(

            filtered,

            selected_variable,

            selected_chart,

            selected_limit

        )

        render_table(
            filtered,
            selected_variable
        )

        display(
            HTML("""
            <div class="dashboard-footer">
                TECHNOLOGY USAGE INTELLIGENCE
                &nbsp; | &nbsp;
                CYBER NEON ANALYTICS
                &nbsp; | &nbsp;
                DATA EXPLORATION SYSTEM
            </div>
            """)
        )


# ================================================================
# 16. INTERACTIVE EVENT HANDLERS
# ================================================================

for control in [

    variable_widget,

    chart_widget,

    limit_widget,

    device_filter,

    purpose_filter

]:

    control.observe(
        update_dashboard,
        names="value"
    )


# ================================================================
# 17. EXPORT BUTTON
# ================================================================

export_button = widgets.Button(

    description="DOWNLOAD FILTERED CSV",

    button_style="",

    icon="download",

    layout=widgets.Layout(width="230px")

)

export_button.style.button_color = "#00AFC7"

def download_csv(button):

    filtered = get_filtered_data()

    content = filtered.to_csv(
        index=False
    ).encode("utf-8")

    files.download(
        "Technology_Usage_Filtered.csv"
    ) if False else None

    from google.colab import output

    output.eval_js("""
    const a = document.createElement('a');
    a.href = URL.createObjectURL(
        new Blob([], {type: 'text/csv'})
    );
    """)

    files.download(
        file_name
    ) if False else None

    # Create downloadable file through Python
    with open(
        "/content/Technology_Usage_Filtered.csv",
        "w",
        encoding="utf-8"
    ) as f:

        f.write(content.decode("utf-8"))

    files.download(
        "/content/Technology_Usage_Filtered.csv"
    )


export_button.on_click(download_csv)

display(export_button)

# ================================================================
# 18. DISPLAY DASHBOARD
# ================================================================

display(output_area)

update_dashboard()

print("CYBER NEON DASHBOARD SUCCESSFULLY LOADED!")
print("Use the dropdown controls to explore your survey data.")

Saving Technology_Usage_Survey_Responses_English.csv to Technology_Usage_Survey_Responses_English (2).csv
Dataset successfully loaded!
Dataset shape: (55, 6)
Data cleaning completed.
Total respondents: 55


,ID,Device,Purpose,Age,Daily Usage (Hours),Monthly Data (GB)
0,ID001,Smartphone,Entertainment,NaN,8.0,5.0
1,ID002,Smartphone,Communication tool & learning media,18.0,7.0,50.0
2,ID003,"Smartphone, Laptop, Tablet",Entertainment,18.0,6.0,10.0
3,ID004,Smartphone,Communication,18.0,9.0,20.0
4,ID005,Smartphone,Communication,18.0,6.0,NaN


Button(description='DOWNLOAD FILTERED CSV', icon='download', layout=Layout(width='230px'), style=ButtonStyle(b…

Output()

CYBER NEON DASHBOARD SUCCESSFULLY LOADED!
Use the dropdown controls to explore your survey data.
